# Trained V0 / V1 — ONNX CPU export and development verification

**Revision: PARKING-V0-V1-ONNX-CPU-2026-10-31-r1**

Use a **CPU runtime**. No GPU, CUDA extension, training, Core ML, camera access or iPhone installation is needed.

- **V0:** unchanged actual-camera adapter, epoch 4.
- **V1:** selected 2–6 m continuation, block 5. The six-cell candidate is not V1.
- Both take **RGB + calibrated [fx, fy, cx, cy]** and output frozen Urban candidates + trained camera-axis depth in metres.
- Verify all 512 frozen development images. Held-out bodies remain closed. This is numerical export parity, not a new product pass or a ±10 cm requirement.

The final phone package includes both graphs, 16 fixed-input runtime fixtures, and all 512 preprocessed dev inputs plus sparse GT. A lossless storage codec reconstructs exact FP32 tensors; this is not model quantization. The existing app remains unchanged until these packages are reviewed.

**Resume:** on the same VM, rerun a stopped section with the same RUN_ID. Completed graph exports and per-shard comparisons are verified and reused; a partial shard restarts. Runtime/code changes fail closed. A VM deletion loses local progress unless you saved it; this notebook does not claim automatic cross-VM resume. Download or copy final archives before ending the runtime.


## 1. Paths and local logging

Existing archives are read sequentially once, then all inference uses `/content`. Uploaded archives take priority over Drive. Set `USE_DRIVE=False` if both ZIPs are already uploaded. Do not use the full sensor dataset or coverage ZIP.


In [ ]:
# PARKING-V0-V1-ONNX-CPU-2026-10-31-r1 — trained two-input models, CPU only.
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, os, shlex, subprocess, sys
RUN_ID = 'parking_v0_v1_trained_onnx_cpu_r1'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive')
MOBILE = Path('/content/mobile_adas3d')
TASK = Path('/content') / RUN_ID
OUTPUT = TASK / 'output'
DEPS = TASK / 'packages'
LOGS = TASK / 'logs'
SOURCE = Path('/content/Urban_Parking_V0V1_Source')
WEIGHTS = TASK / 'weights'
CAPSULE_SOURCE = Path('/content/parking_v0_dev_inputs.zip')
if not CAPSULE_SOURCE.is_file():
    CAPSULE_SOURCE = BASE/'mobile_adas3d_inputs/parking_v0_dev/epoch4_frozen512_r1/parking_v0_dev_inputs.zip'
PILOT_SOURCE = Path('/content/parking_v1_matched_pilot_results.zip')
if not PILOT_SOURCE.is_file():
    PILOT_SOURCE = BASE/'mobile_adas3d_outputs/parking_v1_matched_pilot/parking_v1_matched_adapter_0_6_r1/portable_progress/parking_v1_matched_pilot_results.zip'
PUBLISH = BASE/'mobile_adas3d_outputs/parking_v0_v1_onnx'/RUN_ID if USE_DRIVE else None
PYTHON = Path(sys.executable)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(DEPS)+os.pathsep+str(MOBILE),
           YOLO_AUTOINSTALL='false', OMP_NUM_THREADS='2',
           OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
READY = False
def run_logged(command, name, cwd=None, allow_failure=False, env=None):
    command = [str(v) for v in command]
    LOGS.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = LOGS/(name+'_'+stamp+'.log')
    print('+', shlex.join(command), flush=True)
    print('LOCAL log:', log, flush=True)
    tail = deque(maxlen=30)
    with log.open('w') as stream:
        p = subprocess.Popen(command, cwd=cwd, env=env or ENV,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in p.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush()
                tail.append(line.rstrip())
            status = p.wait()
        except BaseException:
            p.terminate()
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired: p.kill(); p.wait()
            raise
        finally: p.stdout.close()
        stream.write(f'\nEXIT CODE: {status}\n')
    if status and not allow_failure:
        raise RuntimeError(f'Exit {status}; inspect {log}\n'+'\n'.join(tail))
    return status, log
print('CPU-only output:', OUTPUT)
print('V0 capsule:', CAPSULE_SOURCE)
print('Matched pilot:', PILOT_SOURCE)


## 2. Update main and install task-local packages

This does not replace Colab's system PyTorch or change any completed experiment. Matching CPU wheels are installed only inside this task's package folder when needed. A fresh task directory is required if recorded code/runtime changes after export.


In [ ]:
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.is_dir(): run_logged(['git','clone',url,MOBILE], 'clone')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use the project main branch; current branch is {branch!r}. No checkout/reset was attempted.')
run_logged(['git','pull','--ff-only'], 'update_main', MOBILE)
CONFIG = MOBILE/'configs/parking_v0_v1_onnx_v1.json'
SCRIPT = MOBILE/'scripts/export_parking_v0_v1_onnx.py'
for path in (CONFIG,SCRIPT):
    if not path.is_file(): raise FileNotFoundError(f'Update main; missing {path}')
C = json.loads(CONFIG.read_text())
print('Loaded revision:', C['revision'])
if C['revision'] != 'PARKING-V0-V1-ONNX-CPU-2026-10-31-r1':
    raise RuntimeError('Unexpected workflow revision')
packages = [name+'=='+version for name,version in C['packages'].items()]
pip_env = os.environ.copy(); pip_env.pop('PYTHONPATH', None)
run_logged([PYTHON,'-m','pip','install','-q','--no-deps','--upgrade','--target',DEPS,*packages],
           'pinned_packages', MOBILE, env=pip_env)
torch_probe = "import torch,importlib.metadata as m; print(str(torch.__version__).split('+')[0]); print(m.version('torchvision').split('+')[0])"
versions = subprocess.check_output([str(PYTHON),'-c',torch_probe],env=ENV,text=True).strip().splitlines()
if versions != [C['torch_base_version'],C['torchvision_base_version']]:
    run_logged([PYTHON,'-m','pip','install','-q','--no-deps','--upgrade','--target',DEPS,
        '--index-url','https://download.pytorch.org/whl/cpu',
        'torch=='+C['torch_base_version'],'torchvision=='+C['torchvision_base_version']],
        'task_local_torch_cpu',MOBILE,env=pip_env)
probe = "from scripts.export_parking_v0_v1_onnx import configuration,runtime; import json; print(json.dumps(runtime(configuration()),indent=2))"
run_logged([PYTHON,'-c',probe], 'cpu_runtime', MOBILE)


## 3. Stage two verified archives and run tests

If Drive reports an input/output error, stop repeated reads. Upload the same ZIP to `/content`, update the path in section 1, and rerun. No automatic remount or quota bypass.


In [ ]:
sys.path.insert(0,str(MOBILE))
from scripts.parking_v0_local_io import copy_once
CAPSULE = TASK/'archives/parking_v0_dev_inputs.zip'
PILOT = TASK/'archives/parking_v1_matched_pilot_results.zip'
for source,destination,digest in (
    (CAPSULE_SOURCE,CAPSULE,C['capsule_sha256']),
    (PILOT_SOURCE,PILOT,C['pilot_sha256'])):
    if not destination.is_file() and not source.is_file():
        raise FileNotFoundError(f'Upload the existing archive or correct section 1: {source}')
    copy_once(source,destination,digest)
    print('Local archive verified:',destination)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_parking_v0_v1_onnx.py','-v'],
           'export_tests',MOBILE)
def stage(name):
    if not READY: raise RuntimeError('Run sections 1–3 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,
        '--capsule',CAPSULE,'--pilot',PILOT,'--output',OUTPUT,
        '--source',SOURCE,'--weights',WEIGHTS],name,MOBILE)
READY = True


## 4. Export the exact two trained models

Only the existing pinned Urban checkpoint may be downloaded. No new optimizer, training step, model simplification, quantization or resolution change is applied. Both graphs must expose `image` and `camera` inputs.


In [ ]:
stage('export')
print('Both ONNX exports completed. Continue to full development verification.')


## 5. Verify all 512 development images

Compare full candidate/depth outputs, sampled sparse depths, numerical coverage and all range/class MAE cells against the reviewed native endpoints. Progress prints per completed shard. Rerunning verifies and reuses complete shards on this VM.


In [ ]:
stage('validate')
REPORT = OUTPUT/'onnx_comparison.json'
result = json.loads(REPORT.read_text())
assert result['complete'] and result['full_output_parity_passed'] and result['reviewed_point_parity_passed']
assert result['development_images']==512 and result['heldout_bodies_opened']==0
for role in ('V0','V1'):
    current = result['metrics'][role+'_onnx']
    print(role, 'ONNX dev primary MAE (m):',current['conditional_equal_four_cell_mae_m'])
print('No iPhone latency has been measured by this notebook.')


## 6. Build and save review/phone packages

The packaging step also checks that a changed camera calibration changes depth, not detection. The small results ZIP is for review; the larger phone ZIP contains both graphs and all fixed-input/accuracy replay assets (bounded to 1 GB). Save both. This does not install or replace the iPhone app.


In [ ]:
stage('bundle')
RESULTS = OUTPUT/'portable/parking_v0_v1_onnx_results.zip'
PHONE = OUTPUT/'portable/parking_v0_v1_phone.zip'
for path in (RESULTS,PHONE):
    if not path.is_file(): raise FileNotFoundError(path)
    print('READY:',path, f'({path.stat().st_size:,} bytes)')
if PUBLISH is not None:
    from scripts.parking_edge_common import file_sha
    from scripts.export_parking_v0_v1_onnx import copy_published_archive
    for path in (RESULTS,PHONE):
        try:
            copy_published_archive(path,PUBLISH/path.name,file_sha(path))
            print('Saved:',PUBLISH/path.name)
        except (OSError,RuntimeError) as error:
            print('Drive save failed; local output is preserved. Download before closing:',path,error)
print('Return parking_v0_v1_onnx_results.zip for review. Keep the phone ZIP for device integration.')
# Optional manual download:
# from google.colab import files
# files.download(str(RESULTS))
# files.download(str(PHONE))
